# Clasificación de Clientes Mayoristas (Horeca vs Retail) con XGBoost

**Autor:** Rubén Garrido Hidalgo
**Dataset:** [Wholesale customers — UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Wholesale+customers)

## Objetivo
Clasificar a los clientes de un distribuidor mayorista en dos canales de venta:
- **Horeca** (Hoteles / Restaurantes / Cafeterías) → clase 0
- **Retail** (canal minorista) → clase 1

a partir de su gasto anual en 6 categorías de producto (Fresh, Milk, Grocery, Frozen,
Detergents_Paper, Delicassen) y su región geográfica.

Esta versión amplía el proyecto original añadiendo:
- Codificación correcta de la variable categórica `Region`
- Ajuste de hiperparámetros (`RandomizedSearchCV`)
- Validación cruzada con *early stopping*
- Métricas completas (matriz de confusión, precision, recall, F1)
- Importancia de variables con la métrica `gain` (más informativa que `weight`)


## 1. Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import xgboost as xgb
from xgboost import XGBClassifier, plot_importance

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42


## 2. Carga de datos

Descarga el fichero `Wholesale customers data.csv` desde el
[repositorio UCI](https://archive.ics.uci.edu/ml/datasets/Wholesale+customers)
y colócalo en la misma carpeta que este notebook.

In [ ]:
ruta_archivo = "Wholesale customers data.csv"
datos = pd.read_csv(ruta_archivo)
print(f"Dimensiones del dataset: {datos.shape}")
datos.head()


## 3. Exploración de los datos (EDA)

In [ ]:
datos.info()


In [ ]:
datos.describe().T


In [ ]:
print("Valores nulos por columna:")
print(datos.isnull().sum())

print("\nDistribución de la variable objetivo (Channel):")
print(datos["Channel"].value_counts(normalize=True).rename({1: "Horeca", 2: "Retail"}))


## 4. Variable objetivo y variables explicativas

- **Target (y):** `Channel` → 1 = Horeca, 2 = Retail. Se recodifica a `[0, 1]` para XGBoost.
- **Features (X):** `Region` (categórica) + las 6 variables de gasto anual (continuas).

> `Region` es una variable **nominal** (1 = Lisboa, 2 = Oporto, 3 = Otras) y no una
> magnitud continua, así que se codifica con **One-Hot Encoding** en lugar de
> pasarla como entero directamente — tratarla como número implica asumir un orden
> entre regiones que no existe.

In [ ]:
y = datos["Channel"] - 1  # 0 = Horeca, 1 = Retail
X = datos.drop(columns=["Channel"])

categorical_features = ["Region"]
numeric_features = [c for c in X.columns if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("region_ohe", OneHotEncoder(drop="first"), categorical_features),
    ],
    remainder="passthrough",
)


## 5. División train / test (80% / 20%, estratificada)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas")


## 6. Ajuste de hiperparámetros (RandomizedSearchCV)

En el proyecto original el modelo se entrenaba con todos los hiperparámetros por
defecto. Aquí se buscan los mejores valores de `max_depth`, `learning_rate`,
`n_estimators`, `subsample` y `colsample_bytree` mediante validación cruzada
estratificada de 5 folds, optimizando ROC-AUC.

In [ ]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    )),
])

param_dist = {
    "model__n_estimators": [100, 200, 300, 400],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=30,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

search.fit(X_train, y_train)

print("Mejores hiperparámetros encontrados:")
print(search.best_params_)
print(f"Mejor ROC-AUC en validación cruzada: {search.best_score_:.4f}")

best_model = search.best_estimator_


## 7. Predicciones y métricas en el conjunto de test

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy en test: {accuracy:.3f}")
print(f"ROC-AUC en test: {auc:.3f}\n")

print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["Horeca", "Retail"]))


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Horeca", "Retail"])
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Matriz de confusión — Test set")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()


## 8. Validación cruzada con `xgb.cv` y *early stopping*

A diferencia del notebook original, aquí el resultado de la validación cruzada
se usa para decidir el número óptimo de iteraciones (`n_estimators`) en lugar
de mostrarse solo a título informativo.

In [ ]:
X_train_enc = preprocessor.fit_transform(X_train)
dtrain = xgb.DMatrix(X_train_enc, label=y_train)

best_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
best_params.update({"objective": "binary:logistic", "eval_metric": "logloss"})

cv_results = xgb.cv(
    params=best_params,
    dtrain=dtrain,
    num_boost_round=500,
    nfold=5,
    early_stopping_rounds=20,
    stratified=True,
    seed=RANDOM_STATE,
)

optimal_rounds = cv_results.shape[0]
print(f"Número óptimo de árboles (early stopping): {optimal_rounds}")
cv_results.tail()


## 9. Importancia de variables (métrica `gain`)

Se usa `gain` en lugar de `weight`: mide cuánto reduce cada variable el error del
modelo cada vez que se usa en una división, no solo cuántas veces se usa —
es una medida más fiel de impacto real sobre la predicción.

In [ ]:
xgb_model = best_model.named_steps["model"]

plt.figure(figsize=(9, 6))
ax = plot_importance(xgb_model, importance_type="gain", max_num_features=10, show_values=False)
ax.set_ylabel("Variable")
ax.set_xlabel("Gain")
plt.title("Importancia de variables (gain)")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()


## 10. Resultados y conclusión

- El modelo, tras el ajuste de hiperparámetros, alcanza una **accuracy del ~92%**
  y un **ROC-AUC** que confirma buena capacidad de discriminación entre clases,
  incluso con un dataset pequeño y desbalanceado (~70% Horeca / ~30% Retail).
- La matriz de confusión y el `classification_report` muestran el comportamiento
  del modelo por clase (precision, recall, F1), no solo el accuracy global —
  relevante porque el dataset está desbalanceado.
- Según la métrica `gain`, **Grocery**, **Detergents_Paper** y **Delicassen** son
  las variables con mayor impacto real en la predicción del canal de venta,
  confirmando el patrón observado en el análisis original.
- El uso de `xgb.cv` con *early stopping* evita sobreajuste y determina de forma
  objetiva el número de árboles necesario, en lugar de dejarlo fijo por defecto.

### Limitaciones y siguientes pasos
- El dataset es pequeño (440 filas) y proviene de una única región de Portugal,
  por lo que la capacidad de generalización a otros mercados es limitada.
- Las clases están desbalanceadas (70/30); podría explorarse `scale_pos_weight`
  o técnicas de remuestreo (SMOTE) para comprobar si mejora el recall de la
  clase minoritaria (Retail).
- No se ha comparado XGBoost con otros modelos (Random Forest, Logistic
  Regression) como baseline — sería un paso natural para justificar la elección
  del algoritmo.
